# Checkpoint 4E — Satellite Availability Audit + One-Satellite Pilot Comparison

**Question:** *can the existing monthly gridding + threshold pipeline be applied to another comparable
POES/MetOp satellite for the same month/channel/region, and do the threshold-defined footprints remain
broadly similar?* This is a **pilot compatibility check**, NOT a full multi-satellite study and NOT a
calibrated physical comparison.

- Month: January 2024 · Region: lat[-70,20] x lon[-100,20] · lon [0,360)->[-180,180)
- Channel: `mep_omni_flux_p1` (differential proton flux ~25 MeV, `#/cm2-s-str-MeV` if confirmed)
- Grids 5deg/2deg · stats mean_flux, median_flux · thresholds top 20/10/5/2/1%
- Calibration: drop `mep_IFC_on == 1`; keep `== -1` (uninterpreted)

**Caveats:** absolute flux is *not* comparable across satellites without calibration — compare
footprint location/shape first. Differences may be instrument/orbit/coverage, not physical. No SAA
boundary / dose / health / danger / discovery claims.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from saa.satellite_analysis import (
    CANDIDATE_SATELLITES, REFERENCE_SATELLITE, build_satellite_audit, choose_pilot,
    run_satellite_sensitivity, save_table,
    plot_satellite_mean_map, plot_satellite_sample_count, plot_satellite_comparison,
    plot_satellite_centroid_comparison, plot_satellite_area_by_threshold,
)
from saa.aggregate import load_range, build_grid_table, add_coverage_mask
from saa.grid_flux import prepare_region
from saa.load_poes import download_poes_file
from saa.threshold_analysis import haversine_km

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TBL = ROOT / "outputs" / "tables"
FIG = ROOT / "outputs" / "figures"
for d in (PROC, TBL, FIG):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 30)
print("ROOT:", ROOT, "| candidates:", CANDIDATE_SATELLITES)

ROOT: /home/fellipe/Projects/Active/saa-poes-mapping | candidates: ['noaa15', 'noaa18', 'noaa19', 'metop01', 'metop03']


## 1. Satellite availability audit (real NCEI archive paths + real sample-file checks)

In [2]:
audit = build_satellite_audit(CANDIDATE_SATELLITES, raw_dir=str(RAW))
save_table(audit, TBL / "cp4e_satellite_availability_audit.csv",
           TBL / "cp4e_satellite_availability_audit.parquet")
print(audit[["satellite","available_days","missing_days","opens_with_loader","has_time_lat_lon",
             "has_mep_omni_flux_p1","has_mep_IFC_on","channel_units","recommended_for_pilot"]].to_string(index=False))
print("\narchive paths + compatibility notes:")
for _, r in audit.iterrows():
    print(f"  {r.satellite}: {r.archive_path}")
    print(f"      long_name={r.channel_long_name!r}")
    print(f"      note: {r.compatibility_note}")

satellite  available_days missing_days  opens_with_loader  has_time_lat_lon  has_mep_omni_flux_p1  has_mep_IFC_on   channel_units recommended_for_pilot
  metop01              31         none               True              True                  True            True #/cm2-s-str-MeV              eligible
  metop03              31         none               True              True                  True            True #/cm2-s-str-MeV              eligible
   noaa15              31         none               True              True                  True            True #/cm2-s-str-MeV              eligible
   noaa18              31         none               True              True                  True            True #/cm2-s-str-MeV              eligible
   noaa19              31         none               True              True                  True            True #/cm2-s-str-MeV             reference

archive paths + compatibility notes:
  metop01: https://www.ncei.noaa.gov/data/poes-met

## 2. Pilot satellite selection

Selection criteria: complete Jan-2024 coverage, compatible NetCDF variable names, `mep_omni_flux_p1`
present, channel metadata/units matching NOAA-19, minimal missing days, simplest cautious comparison.

In [3]:
pilot = choose_pilot(audit)
print("CHOSEN PILOT:", pilot)
assert pilot is not None, "no eligible pilot satellite found - stop and report (do not force comparison)"
sel = audit[audit.satellite == pilot].iloc[0]
print(f"why {pilot}: {sel.compatibility_note}")
print("rationale: NOAA-18 is the closest NOAA-19 analog - same POES series and SEM-2/MEPED instrument")
print("generation - so it is the simplest, most defensible first cautious comparison. (MetOp = different")
print("EUMETSAT platform; NOAA-15 = oldest, most degraded.) All five were eligible; one pilot only.")

CHOSEN PILOT: noaa18
why noaa18: full Jan coverage; p1 units+long_name match NOAA-19
rationale: NOAA-18 is the closest NOAA-19 analog - same POES series and SEM-2/MEPED instrument
generation - so it is the simplest, most defensible first cautious comparison. (MetOp = different
EUMETSAT platform; NOAA-15 = oldest, most degraded.) All five were eligible; one pilot only.


## 3. Process the pilot satellite (real January 2024 files) + reuse NOAA-19 CP4A products

In [4]:
# robust pre-download of the pilot's 31 daily files (retry transient SSL/network errors)
import urllib.error
days = pd.date_range("2024-01-01", "2024-01-31", freq="D")
ok = 0
for d in days:
    for attempt in range(4):
        try:
            download_poes_file(d.date(), pilot, output_dir=str(RAW)); ok += 1; break
        except (urllib.error.URLError, urllib.error.HTTPError, OSError) as e:
            if attempt == 3:
                print(f"  {d.date()} FAILED: {e}")
print(f"pilot daily files available locally: {ok}/31")

rl = load_range("2024-01-01", "2024-01-31", satellite=pilot, raw_dir=str(RAW))
print(f"{pilot}: loaded {rl.n_loaded}/31 days; missing: {rl.missing_dates or 'none'}")
pilot_region, counts = prepare_region(rl.df)
print(f"{pilot} region rows: {len(pilot_region):,} | counts: {counts}")
pilot_parq = PROC / f"cp4e_{pilot}_2024-01_mep_omni_flux_p1_region.parquet"
pilot_region.to_parquet(pilot_parq, index=False)
print("saved", pilot_parq.name)

pilot daily files available locally: 31/31


noaa18: loaded 31/31 days; missing: none


noaa18 region rows: 208,722 | counts: {'n_total': 1217377, 'n_after_geo': 208722, 'n_ifc_on_dropped': 0, 'n_ifc_minus1': 195680, 'n_after_ifc': 208722}


saved cp4e_noaa18_2024-01_mep_omni_flux_p1_region.parquet


## 4. Grids for the pilot satellite (report sample-count distribution; justify coverage threshold)

In [5]:
# NOAA-19 reference grids from CP4A (no regeneration)
n19_5 = pd.read_parquet(TBL / "cp4a_noaa19_2024-01_grid_5deg.parquet")
n19_2 = pd.read_parquet(TBL / "cp4a_noaa19_2024-01_grid_2deg.parquet")

COVERAGE_THRESHOLD = 30  # same as CP4A (both are full months); justified below after the distribution
pilot_grids = {}
for gd in (5.0, 2.0):
    mask_col = f"enough_samples_{int(gd)}deg"
    t = build_grid_table(pilot_region, lat_step=gd, lon_step=gd)
    t = add_coverage_mask(t, COVERAGE_THRESHOLD, mask_col)
    pilot_grids[int(gd)] = t
    t.to_parquet(TBL / f"cp4e_{pilot}_2024-01_grid_{int(gd)}deg.parquet", index=False)

print("== sample_count distribution (populated cells) ==")
for gd, n19 in [(5, n19_5), (2, n19_2)]:
    p = pilot_grids[gd]["sample_count"]; r = n19["sample_count"]
    pm = f"enough_samples_{gd}deg"
    print(f"  {gd}deg  NOAA-19: n={len(r)} med={r.median():.0f} min={r.min()} max={r.max()} "
          f"pass>=30={int((r>=30).sum())}")
    print(f"        {pilot}: n={len(p)} med={p.median():.0f} min={p.min()} max={p.max()} "
          f"pass>=30={int(pilot_grids[gd][pm].sum())}")
print("\nThreshold justification: both satellites are full 31-day months with comparable per-cell")
print("sampling, so the CP4A >=30 mask is kept for BOTH (a fair, fixed methodological constant).")
print("Coverage is reported above so any difference is visible, not hidden.")

== sample_count distribution (populated cells) ==
  5deg  NOAA-19: n=432 med=477 min=276 max=599 pass>=30=432
        noaa18: n=432 med=487 min=236 max=663 pass>=30=432
  2deg  NOAA-19: n=2700 med=76 min=16 max=127 pass>=30=2685
        noaa18: n=2699 med=78 min=1 max=140 pass>=30=2571

Threshold justification: both satellites are full 31-day months with comparable per-cell
sampling, so the CP4A >=30 mask is kept for BOTH (a fair, fixed methodological constant).
Coverage is reported above so any difference is visible, not hidden.


## 5. Threshold sensitivity for NOAA-19 + pilot (2 sat x 2 grid x 2 stat x 5 thresh = 40 rows)

In [6]:
compat_note = (f"pilot={pilot} vs ref={REFERENCE_SATELLITE}; same SEM-2/MEPED p1 units+long_name; "
               "absolute flux NOT cross-calibrated - compare footprint location/shape only; "
               "mep_IFC_on==-1 uninterpreted")
specs = [
    {"satellite": REFERENCE_SATELLITE, "note": compat_note, "grids": [
        {"grid_deg": 5, "step": 5.0, "table": n19_5, "mask_col": "enough_samples_5deg", "coverage_threshold": 30},
        {"grid_deg": 2, "step": 2.0, "table": n19_2, "mask_col": "enough_samples_2deg", "coverage_threshold": 30}]},
    {"satellite": pilot, "note": compat_note, "grids": [
        {"grid_deg": 5, "step": 5.0, "table": pilot_grids[5], "mask_col": "enough_samples_5deg", "coverage_threshold": 30},
        {"grid_deg": 2, "step": 2.0, "table": pilot_grids[2], "mask_col": "enough_samples_2deg", "coverage_threshold": 30}]},
]
sens = run_satellite_sensitivity(specs)
save_table(sens, TBL / "cp4e_satellite_pilot_threshold_sensitivity.csv",
           TBL / "cp4e_satellite_pilot_threshold_sensitivity.parquet")
print("rows:", len(sens), "| satellites:", sorted(sens.satellite.unique()),
      "| selected_cell_count min:", int(sens.selected_cell_count.min()))
view = sens[(sens.grid_deg == 5) & (sens.statistic_used == "mean_flux")]
print(view[["satellite","threshold_label","cells_available_after_coverage_mask","selected_cell_count",
            "selected_area_km2","centroid_lat_flux_weighted","centroid_lon_flux_weighted","peak_flux"]].to_string(index=False))

rows: 40 | satellites: ['noaa18', 'noaa19'] | selected_cell_count min: 5
satellite threshold_label  cells_available_after_coverage_mask  selected_cell_count  selected_area_km2  centroid_lat_flux_weighted  centroid_lon_flux_weighted  peak_flux
   noaa19           top20                                  432                   87       2.499791e+07                  -20.168561                  -55.266002  31.462024
   noaa19           top10                                  432                   44       1.261403e+07                  -21.154234                  -55.635093  31.462024
   noaa19            top5                                  432                   22       6.309309e+06                  -21.574758                  -55.756577  31.462024
   noaa19            top2                                  432                    9       2.574264e+06                  -22.031485                  -55.320518  31.462024
   noaa19            top1                                  432               

## 6. Inter-satellite footprint comparison (location/shape first; absolute flux treated cautiously)

In [7]:
def fw(sat, thr, gd=5, stat="mean_flux"):
    r = sens[(sens.satellite==sat)&(sens.grid_deg==gd)&(sens.statistic_used==stat)&(sens.threshold_label==thr)].iloc[0]
    return r.centroid_lat_flux_weighted, r.centroid_lon_flux_weighted, r.selected_area_km2, r.peak_flux

for thr in ("top10", "top5"):
    a = fw(REFERENCE_SATELLITE, thr); b = fw(pilot, thr)
    dkm = haversine_km(a[0], a[1], b[0], b[1])
    da = abs(a[2]-b[2]); daf = da/a[2]*100
    print(f"== {thr} (5deg mean) ==")
    print(f"  NOAA-19 centroid ({a[0]:.2f},{a[1]:.2f})  area {a[2]/1e6:.2f} Mkm2  peak {a[3]:.1f}")
    print(f"  {pilot:7s} centroid ({b[0]:.2f},{b[1]:.2f})  area {b[2]/1e6:.2f} Mkm2  peak {b[3]:.1f}")
    print(f"  -> centroid distance {dkm:.0f} km | area diff {da/1e6:.2f} Mkm2 ({daf:.1f}%)")
print("\nNOTE: peak/absolute flux differences are NOT cross-calibrated and may reflect instrument")
print("degradation/response, orbit/local-time sampling, or coverage - not necessarily physical change.")

== top10 (5deg mean) ==
  NOAA-19 centroid (-21.15,-55.64)  area 12.61 Mkm2  peak 31.5
  noaa18  centroid (-21.18,-55.76)  area 12.61 Mkm2  peak 36.2
  -> centroid distance 13 km | area diff 0.00 Mkm2 (0.0%)


== top5 (5deg mean) ==
  NOAA-19 centroid (-21.57,-55.76)  area 6.31 Mkm2  peak 31.5
  noaa18  centroid (-21.58,-55.95)  area 6.31 Mkm2  peak 36.2
  -> centroid distance 20 km | area diff 0.00 Mkm2 (0.0%)

NOTE: peak/absolute flux differences are NOT cross-calibrated and may reflect instrument
degradation/response, orbit/local-time sampling, or coverage - not necessarily physical change.


## 7. Figures (no smoothing/interpolation; satellite + threshold/stat/grid labelled; blank = no data/masked)

In [8]:
# A. mean maps 5deg ; B. sample-count maps 5deg
plot_satellite_mean_map(n19_5, 5.0, "enough_samples_5deg", FIG / "cp4e_noaa19_2024-01_mean_flux_5deg.png", "noaa19")
plot_satellite_mean_map(pilot_grids[5], 5.0, "enough_samples_5deg", FIG / f"cp4e_{pilot}_2024-01_mean_flux_5deg.png", pilot)
plot_satellite_sample_count(n19_5, 5.0, FIG / "cp4e_noaa19_2024-01_sample_count_5deg.png", "noaa19")
plot_satellite_sample_count(pilot_grids[5], 5.0, FIG / f"cp4e_{pilot}_2024-01_sample_count_5deg.png", pilot)
# C. top10 comparison 5deg + 2deg
plot_satellite_comparison([("noaa19", n19_5), (pilot, pilot_grids[5])], 5.0, "enough_samples_5deg",
                          FIG / "cp4e_satellite_comparison_top10_5deg_mean.png", "top10")
plot_satellite_comparison([("noaa19", n19_2), (pilot, pilot_grids[2])], 2.0, "enough_samples_2deg",
                          FIG / "cp4e_satellite_comparison_top10_2deg_mean.png", "top10")
# D. centroid comparison ; E. area by threshold
plot_satellite_centroid_comparison(sens, FIG / "cp4e_satellite_centroid_comparison.png")
plot_satellite_area_by_threshold(sens, FIG / "cp4e_satellite_area_by_threshold.png")
print("CP4E figures:", sorted(p.name for p in FIG.glob("cp4e_*.png")))

CP4E figures: ['cp4e_noaa18_2024-01_mean_flux_5deg.png', 'cp4e_noaa18_2024-01_sample_count_5deg.png', 'cp4e_noaa19_2024-01_mean_flux_5deg.png', 'cp4e_noaa19_2024-01_sample_count_5deg.png', 'cp4e_satellite_area_by_threshold.png', 'cp4e_satellite_centroid_comparison.png', 'cp4e_satellite_comparison_top10_2deg_mean.png', 'cp4e_satellite_comparison_top10_5deg_mean.png']


## 8. Summary

- **Audit:** five satellites have complete Jan-2024 L1b coverage in the real NCEI archive
  (noaa15, noaa18, noaa19, metop01, metop03); all open with the CP2 loader and carry `mep_omni_flux_p1`
  with **identical units/long_name** to NOAA-19. Audit table saved.
- **Pilot:** **NOAA-18** — closest NOAA-19 analog (same POES series + SEM-2/MEPED generation), full
  coverage, identical variable names/metadata. One pilot only; this is not yet a multi-satellite study.
- **Footprint consistency:** the NOAA-18 candidate high-flux footprint **broadly overlaps** NOAA-19;
  top10/top5 flux-weighted centroids are within the distances printed in section 6, with comparable
  selected areas — *inter-satellite footprint consistency* in location/shape.
- **Absolute flux** (peak, intensity) is **not cross-calibrated** and is reported only with caveats;
  any difference may be instrument/orbit/coverage, not physical.

This is a **calibration-limited pilot compatibility check**, *not* a true SAA boundary/center, dose,
health risk, danger zone, or discovery. `mep_IFC_on == -1` retained, uninterpreted.